In [ ]:
# 3_segmenting_epochs.ipynb
#  Cell tag parameters (for batch processing — papermill reads these and loops through the participants)
# PAPERMILL_RUN is toggled "True" if batch processing is being used
# Handles two different pipelines: the baseline check and the standard one)
p_id = 'participant_1'
PAPERMILL_RUN = False
IS_BASELINE_CHECK = False

In [ ]:
# Imports
import sys
from IPython.display import display 
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from autoreject import AutoReject

# Allow imports to work regardless of the notebook's location by locating the project root.
# EEG_DIR is defined in config.py
def find_project_root(current_path, marker='config.py'):
    path = Path(current_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path

root_dir = find_project_root(Path.cwd())
sys.path.append(str(root_dir))

from config import EEG_DIR, BEHAV_DIR, TRIGGER_MAP, conditions

mne.set_log_level('error') # reduce extraneous MNE output

# Plotting Toggle: it is deactivated if running each notebook manually, but if batch processing
# is being used, it is activated so windows don't pop-up
if PAPERMILL_RUN:
    # Batch Mode: Suppress pop-ups and draw plots inline
    %matplotlib inline
    mne.viz.set_browser_backend('matplotlib')
    show_plots = False
else:
    # Manual Mode: Allow interactive pop-ups for data exploration
    %matplotlib auto
    show_plots = True

In [ ]:

# Path management
subj_dir = EEG_DIR / p_id
annot_file = subj_dir / f'{p_id}-annot.fif'

In [ ]:
# Epoching settings

if IS_BASELINE_CHECK:
    # Baseline Check mode
    tmin =  -1.2 # 1.2 s before the trigger
    tmax =  0  # the trigger
    baseline = (-1.2, -1) # corrects -1 to 0 s
    save_dir = subj_dir / 'baseline_check'
    suffix = '-bc-epo.fif'

else:
    # Standard pipeline
    tmin, tmax = -0.200, 1.000
    baseline = (None, 0)
    save_dir = subj_dir
    suffix = '-epo.fif'

save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load Filtered raw data
raw_filt = mne.io.read_raw_fif(subj_dir / f'{p_id}-filt-raw.fif', preload=True)
raw_filt.set_montage('easycap-M1')

In [ ]:
events, event_id = mne.events_from_annotations(raw_filt)
print(f"✅ Found {len(events)} events in the raw file.")

In [ ]:
# Retrieve the annotated file from 2_removing_artifacts.ipynb
if annot_file.exists():
    print(f"✅ Found artifact labels for {p_id}...")
    bad_annot = mne.read_annotations(annot_file)
    raw_filt.set_annotations(raw_filt.annotations + bad_annot)
else:
    print(f"⚠️ WARNING: No annotation file found for {p_id}!")

In [ ]:
events[:10]

In [ ]:
# Verify the expected number of Condition B triggers
# (some participant recordings include a training trigger, some don't — exactly one).
# The training was a Condition B sentence, so the only expected anomaly is one extra B (41 not 40).
expected_count = 40
actual_count = np.sum(events[:, 2] == 1010)

if actual_count == expected_count + 1:
    # Exactly one extra B = the training trial. Relabel the first 1010 as 999 so it's excluded.
    print(f"⚠️ EXTRA TRIGGER: Found {actual_count}. Correcting first as Training...")
    indices_b = np.where(events[:, 2] == 1010)[0]
    events = events.copy()
    events[indices_b[0], 2] = 999

elif actual_count > expected_count + 1:
    # More than one extra B would be bizarre, but it is defensive coding.
    raise ValueError(
        f"{p_id}: {actual_count} Condition B triggers — expected 40 or 41 (one training). "
        f"Investigate before trusting."
    )

elif actual_count < expected_count:
    print(f"⚠️ WARNING: Only found {actual_count} triggers for Condition B.")
    print(f"The pipeline will continue, but you should check {p_id} manually later.")

else:
    print(f"✅ Exact count ({actual_count}) detected. No correction needed.")

In [ ]:
# Align EEG events to the behavioral sequence, attach item labels (so they are available for later steps)
# As said above, the training trial was a Condition B sentence that mistakenly fired the B trigger
# (so only B is ever over-counted). We removed it from the EEG stream (-> 999).
# We also remove the matching training row from the behavioral data, so both sides are
# "real trials only" and the EEG events are a clean contiguous slice of the behavioral
# sequence

df = pd.read_csv(BEHAV_DIR / 'full_merged_data.csv', sep=',', encoding='utf-8', low_memory=False)
subject_num = int(p_id.split('_')[-1])
df = (df[df['Subject'] == subject_num]
        .dropna(subset=['Dur_Word_1'])             # drop phantom E-Prime rows
        .reset_index(drop=True))
df = df[df['Running'] != 'Training'].reset_index(drop=True)   # drop the single training row

# Behavioral trigger sequence (A) and the epoch-able EEG events (B).
# TRIGGER_MAP is defined in config.py
A = df['TrigCode'].map(TRIGGER_MAP).fillna(0).astype(int).to_numpy()
epoch_events = events[np.isin(events[:, 2], [1004, 1010, 1007, 1002])]
B = epoch_events[:, 2].astype(int)

if len(B) > len(A):
    raise ValueError(f"{p_id}: more EEG events ({len(B)}) than behavioral trials ({len(A)}).")

# Scored slide: find the offset where B best overlays a contiguous window of A.
best_off, best_score = max(
    ((off, int(np.sum(A[off:off + len(B)] == B))) for off in range(len(A) - len(B) + 1)),
    key=lambda t: t[1],
)
match_pct = best_score / len(B) * 100
if match_pct < 99.0:
    raise ValueError(f"{p_id}: weak alignment ({match_pct:.1f}%) — investigate before trusting.")
print(f"✅ {p_id}: aligned at offset {best_off}, {best_score}/{len(B)} ({match_pct:.1f}%)")

# Behavioral rows now correspond 1:1, in order, to the epoch-able EEG events.
epoch_metadata = (df.iloc[best_off:best_off + len(B)][['Item', 'Condition']]
                    .reset_index(drop=True))

In [ ]:
event_id

In [ ]:
# Map conditions to events
event_mapping = {'ConditionA': 1004,
                 'ConditionB': 1010,
                 'ConditionC': 1007,
                 'Filler': 1002
                 }

In [ ]:
# Plot events over time
fig, ax = plt.subplots(figsize=[15, 5])

mne.viz.plot_events(events, raw_filt.info['sfreq'], event_id=event_mapping, axes=ax, show=show_plots)
plt.show();

In [ ]:
# Create epochs
epochs = mne.Epochs(raw_filt,
                    epoch_events, event_mapping,
                    tmin, tmax,
                    baseline=baseline,
                    preload=True
                    )
                    
# MNE may drop an event whose window runs off the recording edge (late-start
# recordings — happened with Participant 10 during baseline check). epochs.selection indexes which 
# of epoch_events survived, so we subset the metadata to match before attaching.
# This was added after Participant 10 exposed the edge-drop (it was only raised for them)
epochs.metadata = epoch_metadata.iloc[epochs.selection].reset_index(drop=True)
assert len(epochs.metadata) == len(epochs), \
    f"{p_id}: metadata {len(epochs.metadata)} != epochs {len(epochs)}"

# Visualize epochs
epochs

In [ ]:
# Visualize average ERP across conditions of interest
# conditions is defined in config.py (ConditionA, ConditionB, ConditionC)
epochs[conditions].average().plot(show=show_plots); # remember the semicolon prevents a duplicated plot

In [ ]:
# Read the previously-saved ICA decomposition
ica = mne.preprocessing.read_ica(subj_dir / f'{p_id}-ica.fif')

# Apply the ICA decomposition (excluding the marked ICs) to the epochs
epochs_postica = ica.apply(epochs.copy())

In [ ]:
# Visualize average ERP after ICA Artifact Removal

fig, ax = plt.subplots(1, 2, figsize=[12, 3])

my_ylim = dict(eeg=[-11, 10])

epochs[conditions].average().plot(axes=ax[0], 
    ylim=my_ylim, 
    show=False,
    titles='Before ICA'
);

epochs_postica[conditions].average().plot(
    axes=ax[1], 
    ylim=my_ylim,
    titles='After ICA',
    show=show_plots
);

In [ ]:
# AutoReject for final data cleaning
ar = AutoReject(n_interpolate=[1, 2, 4],
                random_state=42,
                picks=mne.pick_types(epochs_postica.info, 
                                     eeg=True,
                                     eog=False
                                    ),
                n_jobs=1, 
                verbose=False
                )

epochs_clean, reject_log_clean = ar.fit_transform(epochs_postica, return_log=True)

epochs_clean

In [ ]:
assert epochs_clean.metadata is not None and len(epochs_clean.metadata) == len(epochs_clean), \
    f"{p_id}: metadata/epoch length mismatch after AutoReject"
print(f"✅ {len(epochs_clean)} clean epochs, each carrying Item/Condition")

In [ ]:
# View AutoReject's effects

fig, ax = plt.subplots(figsize=[10, 4])
reject_log_clean.plot('horizontal', aspect='auto', ax=ax)
plt.show()

In [ ]:
# Visualize average ERP after Autoreject

fig, ax = plt.subplots(1, 2, figsize=[12, 3])

epochs[conditions].average().plot(axes=ax[0], show=False); # remember the semicolon prevents a duplicated plot
epochs_clean[conditions].average().plot(axes=ax[1], show=show_plots);

In [ ]:
# Scalp topography maps
# Specify times to plot at, as [min],[max],[stepsize]
times = np.arange(tmin, tmax, 0.1)

for cond in conditions:
    fig = epochs_clean[cond].average().plot_topomap(times=times, average=0.050, show=False)
    fig.suptitle(f'{p_id} — {cond}', fontsize=14)
    if show_plots:
        plt.show()
    else:
        display(fig)
        plt.close(fig)

In [ ]:
# Save segmented epochs

epochs_clean.save(save_dir / f'{p_id}{suffix}', overwrite=True);

In [ ]:
print(f"✅ Finished Segmenting for {p_id}")